In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
# Snowflake Python Notebook for Title Cleansing using SQL
import time
import snowflake.snowpark as snowpark


# Define table names
apra_table = "ADC_WORKS_TITLE_VARIANTS"
mzk_table = "MZK_TRACKS_TITLE_VARIANTS"
apra_output_table = "ADC_WORKS_TITLE_VARIANTS_CLEAN"
mzk_output_table = "MZK_TRACKS_TITLE_VARIANTS_CLEAN"

print("Starting title cleansing process...")
start_time = time.time()

# Create JavaScript function for title cleansing
js_cleanse_title = """
CREATE OR REPLACE FUNCTION JS_CLEANSE_TITLE(title VARCHAR)
RETURNS VARCHAR
LANGUAGE JAVASCRIPT
AS
$$
  if (TITLE === null || TITLE === undefined) {
    return "";
  }
  
  // Convert to uppercase
  let result = TITLE.toUpperCase();
  
  // Remove articles from beginning
  result = result.replace(/^(A|AN|THE)\\s+/, '');
  
  // Remove punctuation
  result = result.replace(/[.,\\/#!$%^&*;:{}=\\-_`~()]/g, '');
  
  // Remove extra whitespace
  result = result.replace(/\\s+/g, ' ').trim();
  
  return result;
$$;
"""
session.sql(js_cleanse_title).collect()
print("Created JavaScript cleansing function")

# Process ADC table with SQL
print(f"Cleansing ADC titles and creating {apra_output_table}...")
adc_cleanse_sql = f"""
CREATE OR REPLACE TABLE {apra_output_table} AS
SELECT 
    APRA_WORK_ID,
    ORIGINAL_TITLE,
    JS_CLEANSE_TITLE(TITLE_VARIANT) AS APRA_CLEANED_TITLE,
    ISWC AS APRA_ISWC
FROM {apra_table}
"""
session.sql(adc_cleanse_sql).collect()

# Process MZK table with SQL
print(f"Cleansing MZK titles and creating {mzk_output_table}...")
mzk_cleanse_sql = f"""
CREATE OR REPLACE TABLE {mzk_output_table} AS
SELECT 
    TRACK_ID AS MUZOOKA_TRACK_ID,
    ORIGINAL_TITLE,
    JS_CLEANSE_TITLE(TITLE_VARIANT) AS MUZOOKA_CLEANED_TITLE,
    ISWC AS MUZOOKA_ISWC
FROM {mzk_table}
"""
session.sql(mzk_cleanse_sql).collect()

# Count the number of rows in each cleansed table
adc_count = session.sql(f"SELECT COUNT(*) AS COUNT FROM {apra_output_table}").collect()[0]["COUNT"]
mzk_count = session.sql(f"SELECT COUNT(*) AS COUNT FROM {mzk_output_table}").collect()[0]["COUNT"]

elapsed = time.time() - start_time
print(f"Cleansing completed in {elapsed:.2f} seconds")
print(f"Processed {adc_count} ADC records and {mzk_count} MZK records")

print("Title cleansing completed successfully!")

In [ ]:
CREATE OR REPLACE TABLE FUZZY_TITLE_MATCH AS (
  WITH cte1 AS (
    SELECT distinct * FROM adc_works_non_exact_match_title_variants 
    LIMIT 1000000
  ),
  cte2 AS (
    SELECT * FROM mzk_tracks_title_variants_clean
  )
  SELECT distinct
    cte1.*, 
    UPPER(cte2.muzooka_track_id) AS muzooka_track_id, 
    cte2.muzooka_cleaned_title, 
    cte2.muzooka_iswc,
    jarowinkler_similarity(cte1.apra_cleaned_title, cte2.muzooka_cleaned_title) AS score
  FROM cte1 
  JOIN cte2
  ON SUBSTRING(cte1.apra_cleaned_title, 1, 3) = SUBSTRING(cte2.muzooka_cleaned_title, 1, 3)
  WHERE jarowinkler_similarity(cte1.apra_cleaned_title, cte2.muzooka_cleaned_title) >= 90
);